# 0. Imports, Install (if needed) & Config


Comment the cells out if unnecessary. Make sure you're using the same Python env as your previous notebook.

In [1]:
!pip install pandas pyarrow numpy fastparquet scikit-learn rank-bm25 lightgbm datasets transformers sentencepiece einops
!mkdir -p esci_data
from datasets import load_dataset
print("Downloading ESCI dataset from Hugging Face...")
ds = load_dataset("tasksource/esci")
print("Saving train split...")
ds["train"].to_pandas().to_parquet("esci_data/esci_train.parquet")

print("Saving test split...")
ds["test"].to_pandas().to_parquet("esci_data/esci_test.parquet")

print("Done! Files saved in /content/esci_data/")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 65.2 MB/s eta 0:00:00


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00011-2d36455632bef8(…):   0%|          | 0.00/115M [00:00<?, ?B/s]

data/train-00001-of-00011-18b81793a48399(…):   0%|          | 0.00/120M [00:00<?, ?B/s]

data/train-00002-of-00011-71f741fdff9a6f(…):   0%|          | 0.00/144M [00:00<?, ?B/s]

...

data/test-00002-of-00004-a81cff173329b48(…):   0%|          | 0.00/193M [00:00<?, ?B/s]

data/test-00003-of-00004-22af4ca7fa1313b(…):   0%|          | 0.00/175M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2027874 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/652490 [00:00<?, ? examples/s]

Saving train split...
Saving test split...
Done! Files saved in /content/esci_data/


In [2]:
# Create main folders
!mkdir -p esci_pipeline
!mkdir -p models

# Create subfolders
!mkdir -p esci_pipeline/artifacts_ru
!mkdir -p esci_pipeline/artifacts_us
!mkdir -p esci_pipeline/data
!mkdir -p esci_pipeline/data_ru

# Copy parquet files from root
!cp esci_data/*.parquet esci_pipeline/data/
!cp esci_data/*.parquet esci_pipeline/data_ru/

In [3]:
import sys

print("Python executable:", sys.executable)
sys.path = [p for p in sys.path if "python3.8" not in p]

print("\nFiltered sys.path (no python3.8 entries):")
for p in sys.path:
    print(" ", p)


Python executable: /usr/bin/python3

Filtered sys.path (no python3.8 entries):
  /content
  /env/python
  /usr/lib/python312.zip
  /usr/lib/python3.12
  /usr/lib/python3.12/lib-dynload
  
  /usr/local/lib/python3.12/dist-packages
  /usr/lib/python3/dist-packages
  /usr/local/lib/python3.12/dist-packages/IPython/extensions
  /root/.ipython


In [4]:
import sys
print("Python executable:", sys.executable)

import numpy as np
import pandas as pd
import sklearn
from rank_bm25 import BM25Okapi
import lightgbm as lgb
import torch
import pyarrow
import datasets
import transformers
import huggingface_hub
import tokenizers

print("\nALL CODE-CRITICAL IMPORTS LOADED OK\n")

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("sklearn:", sklearn.__version__)
print("rank_bm25:", BM25Okapi)
print("lightgbm:", lgb.__version__)
print("torch:", torch.__version__)
print("pyarrow:", pyarrow.__version__)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("tokenizers:", tokenizers.__version__)


Python executable: /usr/bin/python3

ALL CODE-CRITICAL IMPORTS LOADED OK

numpy: 2.0.2
pandas: 2.2.2
sklearn: 1.6.1
rank_bm25: <class 'rank_bm25.BM25Okapi'>
lightgbm: 4.6.0
torch: 2.9.0+cu126
pyarrow: 18.1.0
datasets: 4.0.0
transformers: 4.57.2
huggingface_hub: 0.36.0
tokenizers: 0.22.1


In [5]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

from rank_bm25 import BM25Okapi
import lightgbm as lgb

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

# ---- CONFIG ----
DATA_ROOT    = Path("esci_pipeline/data_ru")
PATH_TRAIN   = DATA_ROOT / "esci_train.parquet"
PATH_TEST    = DATA_ROOT / "esci_test.parquet"

LOCALE       = "us"
RANDOM_STATE = 42

# Models
GTE_MODEL_NAME  = "Alibaba-NLP/gte-multilingual-base"
XENC_MODEL_NAME = "distilbert-base-multilingual-cased"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)


# ---- PERSISTENCE CONFIG ----
ARTIFACT_DIR = Path("esci_pipeline/artifacts_ru")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PATH_TRAIN_PREP = ARTIFACT_DIR / "train_df_prepared.parquet"
PATH_VALID_PREP = ARTIFACT_DIR / "valid_df_prepared.parquet"
PATH_PROD_EMBS  = ARTIFACT_DIR / "prod_embs.npy"
PATH_PROD_IDS   = ARTIFACT_DIR / "product_ids.npy"
PATH_BM25_PKL   = ARTIFACT_DIR / "bm25_corpus.pkl"
PATH_LGB_MODEL  = ARTIFACT_DIR / "ltr_model_us.txt"

Using device: cuda


# 1. Load & Basic Cleaning

In [6]:
# 1.1 Load ESCI train/test
esci_train = pd.read_parquet(PATH_TRAIN)
esci_test  = pd.read_parquet(PATH_TEST)

print("Raw shapes:", esci_train.shape, esci_test.shape)
print("Locales train:\n", esci_train["product_locale"].value_counts())

# 1.2 Filter to locale (US for now)
esci_train = esci_train[esci_train["product_locale"] == LOCALE].copy()
esci_test  = esci_test[esci_test["product_locale"] == LOCALE].copy()

# 1.3 Drop rows with missing query/title
esci_train = esci_train.dropna(subset=["query", "product_title"])
esci_test  = esci_test.dropna(subset=["query", "product_title"])

# 1.4 Fill remaining text NaNs
text_cols = [
    "product_description",
    "product_bullet_point",
    "product_brand",
    "product_color",
    "product_text",
]

for col in text_cols:
    esci_train[col] = esci_train[col].fillna("")
    esci_test[col]  = esci_test[col].fillna("")

print("Cleaned shapes:", esci_train.shape, esci_test.shape)

# 1.5 Map ESCI labels to numeric relevance 0–3
label2rel = {
    "Irrelevant": 0,
    "Complement": 1,
    "Substitute": 2,
    "Exact": 3,
}

esci_train["relevance"] = esci_train["esci_label"].map(label2rel).astype("int8")
esci_test["relevance"]  = esci_test["esci_label"].map(label2rel).astype("int8")

print("Relevance distribution (train):")
print(esci_train["relevance"].value_counts().sort_index())


Raw shapes: (2027874, 14) (652490, 14)
Locales train:
 product_locale
us    1420372
jp     333112
es     274390
Name: count, dtype: int64
Cleaned shapes: (1420372, 14) (434234, 14)
Relevance distribution (train):
relevance
0    122273
1     29713
2    280324
3    988062
Name: count, dtype: int64


In [7]:
# ---- Balanced 200k subset of US ESCI (by relevance) ----
DESIRED_TOTAL = 200_000
LABEL_COL = "relevance"

# Check class distribution
label_counts = esci_train[LABEL_COL].value_counts().sort_index()
print("Full US label counts:\n", label_counts)

num_classes = label_counts.shape[0]

# Ideal per-class target (for 200k total)
ideal_per_class = DESIRED_TOTAL // num_classes
print("Ideal per-class target:", ideal_per_class)

# But make sure we don't ask more than the smallest class has
min_available = label_counts.min()
n_per_class = min(ideal_per_class, min_available)

if n_per_class < ideal_per_class:
    print(
        f"⚠ Smallest class only has {min_available} examples; "
        f"using n_per_class={n_per_class} (total={n_per_class * num_classes}) "
        f"instead of full 50,000."
    )

print("Using n_per_class =", n_per_class)

# Sample a balanced subset
esci_train_balanced = (
    esci_train
    .groupby(LABEL_COL, group_keys=False)
    .apply(lambda g: g.sample(n=n_per_class, random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

print("Balanced subset shape:", esci_train_balanced.shape)
print(esci_train_balanced[LABEL_COL].value_counts().sort_index())

# Optionally save for translation / reuse
BALANCED_PATH = DATA_ROOT / "esci_us_balanced_200k.parquet"
esci_train_balanced.to_parquet(BALANCED_PATH, index=False)
print("Saved balanced subset to:", BALANCED_PATH)

esci_train = esci_train_balanced


Full US label counts:
 relevance
0    122273
1     29713
2    280324
3    988062
Name: count, dtype: int64
Ideal per-class target: 50000
⚠ Smallest class only has 29713 examples; using n_per_class=29713 (total=118852) instead of full 50,000.
Using n_per_class = 29713


/tmp/ipython-input-1815475923.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=n_per_class, random_state=RANDOM_STATE))


Balanced subset shape: (118852, 15)
relevance
0    29713
1    29713
2    29713
3    29713
Name: count, dtype: int64
Saved balanced subset to: esci_pipeline/data_ru/esci_us_balanced_200k.parquet


In [8]:
# Export only the fields needed for translation
to_translate = esci_train_balanced[[
    "example_id",
    "query",
    "product_title",
    "product_description",
    "product_bullet_point",
    "product_brand",
    "product_color",
    "relevance",
    "product_id",
    "query_id",
]].copy()

to_translate_path = DATA_ROOT / "us_balanced_200k_for_translation.parquet"
to_translate.to_parquet(to_translate_path, index=False)

print("Saved for translation:", to_translate_path)


Saved for translation: esci_pipeline/data_ru/us_balanced_200k_for_translation.parquet


In [9]:
# ---- Balanced 40k subset of US ESCI TEST (by relevance) ----
DESIRED_TEST = 40_000    # change to 5_000 if you prefer
LABEL_COL_TEST = "relevance"

# Check class distribution in test
label_counts_test = esci_test[LABEL_COL_TEST].value_counts().sort_index()
print("Full US TEST label counts:\n", label_counts_test)

num_classes_test = label_counts_test.shape[0]
ideal_per_class_test = DESIRED_TEST // num_classes_test
print("Ideal per-class target (test):", ideal_per_class_test)

min_available_test = label_counts_test.min()
n_per_class_test = min(ideal_per_class_test, min_available_test)

if n_per_class_test < ideal_per_class_test:
    print(
        f"⚠ Smallest test class only has {min_available_test} examples; "
        f"using n_per_class_test={n_per_class_test} "
        f"(total={n_per_class_test * num_classes_test}) instead of full {DESIRED_TEST}."
    )

print("Using n_per_class_test =", n_per_class_test)

esci_test_balanced = (
    esci_test
    .groupby(LABEL_COL_TEST, group_keys=False)
    .apply(lambda g: g.sample(n=n_per_class_test, random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

print("Balanced TEST subset shape:", esci_test_balanced.shape)
print(esci_test_balanced[LABEL_COL_TEST].value_counts().sort_index())

# Export only the fields needed for translation
test_to_translate = esci_test_balanced[[
    "example_id",
    "query",
    "product_title",
    "product_description",
    "product_bullet_point",
    "product_brand",
    "product_color",
    "relevance",
    "product_id",
    "query_id",
]].copy()

test_to_translate_path = DATA_ROOT / "us_test_balanced_for_translation.parquet"
test_to_translate.to_parquet(test_to_translate_path, index=False)

print("Saved TEST subset for translation:", test_to_translate_path)


Full US TEST label counts:
 relevance
0     43505
1     11147
2     97163
3    282419
Name: count, dtype: int64
Ideal per-class target (test): 10000
Using n_per_class_test = 10000


/tmp/ipython-input-1345452412.py:28: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=n_per_class_test, random_state=RANDOM_STATE))


Balanced TEST subset shape: (40000, 15)
relevance
0    10000
1    10000
2    10000
3    10000
Name: count, dtype: int64
Saved TEST subset for translation: esci_pipeline/data_ru/us_test_balanced_for_translation.parquet


In [10]:
# 1.X Translate balanced US subset to Russian
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the same data we just saved (for reproducibility)
to_translate_path = DATA_ROOT / "us_balanced_200k_for_translation.parquet"
df_src = pd.read_parquet(to_translate_path)
print("Loaded for translation:", df_src.shape)

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MT_MODEL_NAME = "Helsinki-NLP/opus-mt-en-ru"  # EN → RU


mt_tokenizer = AutoTokenizer.from_pretrained(MT_MODEL_NAME)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(
    MT_MODEL_NAME,
    use_safetensors=True,     # <- avoids the torch.load vulnerability path
    torch_dtype=torch.float32 # safe default
).to(DEVICE)
mt_model.eval()

@torch.no_grad()
def translate_batch(texts, max_length=128):
    """
    Translate a list of English strings to Russian.
    Empty strings stay empty.
    """
    if all((t is None or t == "") for t in texts):
        return ["" for _ in texts]

    texts = [(t if t is not None else "") for t in texts]

    enc = mt_tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    ).to(DEVICE)

    gen = mt_model.generate(
        **enc,
        max_length=max_length,
        num_beams=4,
        early_stopping=True,
    )
    out = mt_tokenizer.batch_decode(gen, skip_special_tokens=True)

    if len(out) < len(texts):
        out += [""] * (len(texts) - len(out))
    return out

# ---- Apply translation column-wise ----
cols_to_translate = [
    "query",
    "product_title",
    "product_description",
    "product_bullet_point",
    "product_brand",
    "product_color",
]

df_ru = df_src.copy()
BATCH_SIZE = 32

for col in cols_to_translate:
    print(f"\nTranslating column: {col}")
    src_col = df_src[col].fillna("").astype(str)
    translated = []

    for start in range(0, len(src_col), BATCH_SIZE):
        batch = src_col.iloc[start:start + BATCH_SIZE].tolist()
        ru_batch = translate_batch(batch, max_length=128)
        translated.extend(ru_batch)

        if (start // BATCH_SIZE) % 50 == 0:
            print(f"  processed {start + len(batch)}/{len(src_col)} rows", end="\r")

    df_ru[col] = translated
    print(f"Done: {col}")

# ---- Add locale + label text for RU pipeline compatibility ----
# df_ru["product_locale"] = "ru"
# label_map_inv = {0: "Irrelevant", 1: "Complement", 2: "Substitute", 3: "Exact"}
# df_ru["esci_label"] = df_ru["relevance"].map(label_map_inv)

# ---- Save Russian-translated dataset ----
RU_PATH = DATA_ROOT / "esci_ru_balanced_200k.parquet"
df_ru.to_parquet(RU_PATH, index=False)
print("\nSaved Russian-translated dataset to:", RU_PATH)
# USE THE TRANSLATED RUSSIAN DATA FROM THIS POINT FORWARD
esci_train = df_ru.copy()
print("Shape:", df_ru.shape)
print("Label distribution:\n", df_ru["relevance"].value_counts().sort_index())




Loaded for translation: (118852, 10)


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/803k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/307M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]


Translating column: query
Done: query

Translating column: product_title
Done: product_title

Translating column: product_description
Done: product_description

Translating column: product_bullet_point
Done: product_bullet_point

Translating column: product_brand
Done: product_brand

Translating column: product_color
Done: product_color

Saved Russian-translated dataset to: esci_pipeline/data_ru/esci_ru_balanced_200k.parquet
Shape: (118852, 10)
Label distribution:
 relevance
0    29713
1    29713
2    29713
3    29713
Name: count, dtype: int64


In [11]:
# Adding remaining columns
# 1. Load original balanced US dataset used for translation
us_train_balanced_path = DATA_ROOT / "us_balanced_200k_for_translation.parquet"
df_us_train = pd.read_parquet(us_train_balanced_path)

# 2. Load text-only RU translation (just Russian text)
df_ru_text = esci_train.copy()       # this is df_ru you just created

# Sanity: ensure they have same row count
assert df_us_train.shape[0] == df_ru_text.shape[0], "Row count mismatch!"

# 3. Rebuild complete RU dataset by replacing text columns
cols_to_translate = [
    "query",
    "product_title",
    "product_description",
    "product_bullet_point",
    "product_brand",
    "product_color",
]

df_ru_full = df_us_train.copy()
for col in cols_to_translate:
    df_ru_full[col] = df_ru_text[col].astype(str)

# Add locale + label if needed
df_ru_full["product_locale"] = "ru"
df_ru_full["esci_label"] = df_ru_full["relevance"].map({
    0: "Irrelevant",
    1: "Complement",
    2: "Substitute",
    3: "Exact",
})

print("Rebuilt RU TRAIN shape:", df_ru_full.shape)
print("RU TRAIN columns:", df_ru_full.columns.tolist())

# IMPORTANT: overwrite esci_train with rebuilt dataset
esci_train = df_ru_full


Rebuilt RU TRAIN shape: (118852, 12)
RU TRAIN columns: ['example_id', 'query', 'product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color', 'relevance', 'product_id', 'query_id', 'product_locale', 'esci_label']


In [12]:
# ---- Load the TEST subset we just saved ----
test_to_translate_path = DATA_ROOT / "us_test_balanced_for_translation.parquet"
df_test_src = pd.read_parquet(test_to_translate_path)
print("Loaded TEST subset for translation:", df_test_src.shape)

cols_to_translate = [
    "query",
    "product_title",
    "product_description",
    "product_bullet_point",
    "product_brand",
    "product_color",
]

df_test_ru = df_test_src.copy()
BATCH_SIZE = 32

for col in cols_to_translate:
    print(f"\nTranslating TEST column: {col}")
    src_col = df_test_src[col].fillna("").astype(str)
    translated = []

    for start in range(0, len(src_col), BATCH_SIZE):
        batch = src_col.iloc[start:start + BATCH_SIZE].tolist()
        ru_batch = translate_batch(batch, max_length=128)
        translated.extend(ru_batch)

        # light progress log
        if (start // BATCH_SIZE) % 20 == 0:
            print(f"  processed {start + len(batch)}/{len(src_col)} rows", end="\r")

    df_test_ru[col] = translated
    print(f"Done: {col}")

# ---- Add locale + esci_label for RU test ----
#df_test_ru["product_locale"] = "ru"
#label_map_inv = {0: "Irrelevant", 1: "Complement", 2: "Substitute", 3: "Exact"}
#df_test_ru["esci_label"] = df_test_ru["relevance"].map(label_map_inv)

# ---- Save Russian TEST dataset ----
RU_TEST_PATH = DATA_ROOT / "esci_ru_test_balanced.parquet"
df_test_ru.to_parquet(RU_TEST_PATH, index=False)

print("\nSaved Russian TEST dataset to:", RU_TEST_PATH)
esci_test = df_test_ru.copy()
print("RU TEST shape:", df_test_ru.shape)
print("RU TEST label distribution:\n", df_test_ru["relevance"].value_counts().sort_index())


Loaded TEST subset for translation: (40000, 10)

Translating TEST column: query
Done: query

Translating TEST column: product_title
Done: product_title

Translating TEST column: product_description
Done: product_description

Translating TEST column: product_bullet_point
Done: product_bullet_point

Translating TEST column: product_brand
Done: product_brand

Translating TEST column: product_color
Done: product_color

Saved Russian TEST dataset to: esci_pipeline/data_ru/esci_ru_test_balanced.parquet
RU TEST shape: (40000, 10)
RU TEST label distribution:
 relevance
0    10000
1    10000
2    10000
3    10000
Name: count, dtype: int64


In [13]:
# Load original US TEST balanced dataset (with metadata)
us_test_balanced_path = DATA_ROOT / "us_test_balanced_for_translation.parquet"
df_us_test = pd.read_parquet(us_test_balanced_path)

# Load text-only RU translation for TEST
df_ru_test_text = esci_test.copy()

# Sanity
assert df_us_test.shape[0] == df_ru_test_text.shape[0], "Row mismatch in TEST!"

# Replace only translated text columns
cols_to_translate = [
    "query",
    "product_title",
    "product_description",
    "product_bullet_point",
    "product_brand",
    "product_color",
]

df_ru_test_full = df_us_test.copy()
for col in cols_to_translate:
    df_ru_test_full[col] = df_ru_test_text[col].astype(str)

df_ru_test_full["product_locale"] = "ru"
df_ru_test_full["esci_label"] = df_ru_test_full["relevance"].map({
    0: "Irrelevant",
    1: "Complement",
    2: "Substitute",
    3: "Exact",
})

print("Rebuilt RU TEST shape:", df_ru_test_full.shape)
print("RU TEST columns:", df_ru_test_full.columns.tolist())

# Overwrite esci_test
esci_test = df_ru_test_full


Rebuilt RU TEST shape: (40000, 12)
RU TEST columns: ['example_id', 'query', 'product_title', 'product_description', 'product_bullet_point', 'product_brand', 'product_color', 'relevance', 'product_id', 'query_id', 'product_locale', 'esci_label']


In [14]:
import re

def normalize_ru(text: str) -> str:
    if text is None:
        return ""
    # Lowercase
    text = text.lower()
    # Keep digits, Latin letters, and Cyrillic (а-я ё); remove other junk
    text = re.sub(r"[^0-9a-zа-яё]+", " ", text, flags=re.IGNORECASE)
    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

RU_TEXT_COLS = [
    "query",
    "product_title",
    "product_description",
    "product_bullet_point",
    "product_brand",
    "product_color",
]

for col in RU_TEXT_COLS:
    esci_train[col] = esci_train[col].astype(str).map(normalize_ru)
    esci_test[col]  = esci_test[col].astype(str).map(normalize_ru)


# 2. Unified Text + Explicit Context Features

In [15]:
# 2.1 Unified product_text_clean
def build_product_text_clean(df: pd.DataFrame) -> pd.Series:
    parts = [
        df["product_title"],
        df["product_description"],
        df["product_bullet_point"],
        df["product_brand"],
        df["product_color"],
    ]
    return (
        parts[0].fillna("")
        + " [SEP] " + parts[1].fillna("")
        + " [SEP] " + parts[2].fillna("")
        + " [SEP] " + parts[3].fillna("")
        + " [SEP] " + parts[4].fillna("")
    )

esci_train["product_text_clean"] = build_product_text_clean(esci_train)
esci_test["product_text_clean"]  = build_product_text_clean(esci_test)

# 2.2 Explicit "context" features from metadata
def add_context_features(df: pd.DataFrame) -> pd.DataFrame:
    # --- Basic lengths ---
    df["ctx_title_len"]  = df["product_title"].fillna("").str.split().str.len().astype("float32")
    df["ctx_desc_len"]   = df["product_description"].fillna("").str.split().str.len().astype("float32")
    df["ctx_bullet_len"] = df["product_bullet_point"].fillna("").str.split().str.len().astype("float32")

    # Precompute lowercased strings
    title_lower = df["product_title"].fillna("").str.lower()
    brand_lower = df["product_brand"].fillna("").str.lower()
    color_lower = df["product_color"].fillna("").str.lower()

    # --- Brand in title (row-wise substring check) ---
    df["ctx_brand_in_title"] = [
        int((b != "") and (b in t))
        for b, t in zip(brand_lower, title_lower)
    ]

    # --- Color in title (row-wise substring check) ---
    df["ctx_color_in_title"] = [
        int((c != "") and (c in t))
        for c, t in zip(color_lower, title_lower)
    ]

    # Cast to int8 for compactness
    df["ctx_brand_in_title"] = df["ctx_brand_in_title"].astype("int8")
    df["ctx_color_in_title"] = df["ctx_color_in_title"].astype("int8")

    return df

esci_train = add_context_features(esci_train)
esci_test  = add_context_features(esci_test)

# 2.3 Popularity / behaviour proxies (within ESCI train)
prod_stats = (
    esci_train.groupby("product_id")["relevance"]
    .agg(prod_count="size", prod_mean_rel="mean", prod_max_rel="max")
    .reset_index()
)

query_stats = (
    esci_train.groupby("query_id")["relevance"]
    .agg(query_count="size", query_mean_rel="mean")
    .reset_index()
)

esci_train = esci_train.merge(prod_stats, on="product_id", how="left")
esci_train = esci_train.merge(query_stats, on="query_id", how="left")

# For test, merge stats computed from train only
esci_test = esci_test.merge(prod_stats, on="product_id", how="left")
esci_test = esci_test.merge(query_stats, on="query_id", how="left")

# Fill NaNs for unseen products/queries in test
pop_cols = ["prod_count", "prod_mean_rel", "prod_max_rel", "query_count", "query_mean_rel"]
for col in pop_cols:
    esci_train[col] = esci_train[col].fillna(0).astype("float32")
    esci_test[col]  = esci_test[col].fillna(0).astype("float32")

esci_train.head()


,example_id,query,product_title,product_description,product_bullet_point,product_brand,product_color,relevance,product_id,query_id,...,ctx_title_len,ctx_desc_len,ctx_bullet_len,ctx_brand_in_title,ctx_color_in_title,prod_count,prod_mean_rel,prod_max_rel,query_count,query_mean_rel
0,193260,amazon com код активирован,хулу,я не знаю что делать,получите неограниченный мгновенный доступ к ог...,гулу,я не знаю что делать,0,B0066TUXU6,8677,...,1.0,5.0,65.0,0,0,1.0,0.0,0.0,4.0,0.75
1,264260,браслет для малышки,quot goldenchen mealshion juelry 925 quot сере...,голденхен лидер модных ювелирных изделий котор...,высококачественный медный материал без вредных...,золотая кухня,серебро,0,B07663TBC4,12307,...,17.0,47.0,49.0,0,0,1.0,0.0,0.0,4.0,0.50
2,1311389,макс японский,макс ваимо hd 11flk flat clinch stapler с трем...,я не знаю что делать,качественный плоский степлер для 35 листов бум...,макс япония,разряд,0,B06Y1CBVKT,66517,...,12.0,5.0,28.0,0,0,1.0,0.0,0.0,5.0,0.00
3,1632982,моющее масло под давлением,охранник стирального насоса simpson,я не знаю что делать,предназначенная для установки большинства омыв...,симпсон уборка,оригинал,0,B004S67CUS,83198,...,4.0,5.0,29.0,0,0,1.0,0.0,0.0,2.0,0.00
4,300511,пляжные шляпы для женщин,один кислота боженного винного вина красный l,для того чтобы внушить уверенность и красоту с...,дизайн halter neck loop strap detail ruged sid...,следует,вино красное,0,B072BJPTKJ,14164,...,7.0,60.0,52.0,0,0,1.0,0.0,0.0,5.0,1.80


# 3. Train/Valid Split (Grouped by query_id)

In [16]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=RANDOM_STATE)
train_idx, valid_idx = next(gss.split(esci_train, groups=esci_train["query_id"]))

train_df = esci_train.iloc[train_idx].reset_index(drop=True)
valid_df = esci_train.iloc[valid_idx].reset_index(drop=True)

print("Train / Valid shapes:", train_df.shape, valid_df.shape)


Train / Valid shapes: (107125, 23) (11727, 23)


# 4. BM25 Candidate Generation

In [17]:
def build_bm25_corpus(df: pd.DataFrame):
    # one doc per product_id
    prod_group = df.groupby("product_id")["product_text_clean"].first()
    product_ids = prod_group.index.to_list()
    corpus = [doc.split() for doc in prod_group.values]
    return product_ids, corpus

bm25_product_ids, bm25_corpus = build_bm25_corpus(esci_train)
bm25 = BM25Okapi(bm25_corpus)

# Map product_id -> index in BM25 corpus
prodid_to_idx = {pid: i for i, pid in enumerate(bm25_product_ids)}

def bm25_candidates_for_query(query_text: str, top_k: int = 100):
    tokenized = query_text.split()
    scores = bm25.get_scores(tokenized)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(bm25_product_ids[i], float(scores[i])) for i in top_idx]

# Example sanity check
print(bm25_candidates_for_query(esci_train["query"].iloc[0], top_k=5))



[('B08GD3D9YJ', 20.030306776196227), ('B07741VHLB', 17.961821098191628), ('B07P76HM3B', 17.919597956048662), ('B01FIS7AU6', 17.87757285760172), ('B07X3LSNMQ', 17.87757285760172)]


In [19]:
# ---- SAVE BM25 corpus (optional but useful) ----
import pickle

with open(PATH_BM25_PKL, "wb") as f:
    pickle.dump((bm25_product_ids, bm25_corpus), f)

print("Saved BM25 corpus to:", PATH_BM25_PKL)

Saved BM25 corpus to: esci_pipeline/artifacts_ru/bm25_corpus.pkl


# 5. Attach BM25 Scores & Ranks to train/valid

In [20]:
def attach_bm25_scores(df_queries: pd.DataFrame, top_k: int = 100) -> pd.DataFrame:
    df_queries = df_queries.copy()
    rows = []

    for qid, sub in df_queries.groupby("query_id"):
        q_text = sub["query"].iloc[0]
        cand = bm25_candidates_for_query(q_text, top_k=top_k)
        cand_map = {pid: score for pid, score in cand}
        for idx, row in sub.iterrows():
            pid = row["product_id"]
            bm25_score = cand_map.get(pid, 0.0)
            rows.append((idx, bm25_score))

    bm25_scores = pd.DataFrame(rows, columns=["row_idx", "bm25_score"]).set_index("row_idx")
    df_queries = df_queries.join(bm25_scores, how="left")
    df_queries["bm25_score"] = df_queries["bm25_score"].fillna(0.0)

    # Rank within each query (higher score = better rank)
    df_queries["bm25_rank"] = (
        df_queries.groupby("query_id")["bm25_score"]
        .rank(ascending=False, method="first")
        .astype("int32")
    )
    return df_queries

train_df = attach_bm25_scores(train_df, top_k=100)
valid_df = attach_bm25_scores(valid_df, top_k=100)

train_df[["query_id", "bm25_score", "bm25_rank"]].head()


,query_id,bm25_score,bm25_rank
0,8677,0.000000,1
1,12307,0.000000,1
2,66517,7.621832,1
3,83198,0.000000,1
4,79687,0.000000,1


# 6. GTE Multilingual Embeddings & Similarity

In [21]:
from sentence_transformers import SentenceTransformer
import numpy as np
import torch

# ---- CONFIG ----
GTE_MODEL_NAME = "Alibaba-NLP/gte-multilingual-base"

# Load model on correct device
gte_model = SentenceTransformer(
    GTE_MODEL_NAME,
    trust_remote_code=True,
    device=DEVICE,   # "cuda" or "cpu" from your earlier config
)

@torch.no_grad()
def encode_texts(texts, batch_size=64):
    # SentenceTransformer handles tokenization + model + pooling internally
    embs = gte_model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,    # already L2-normalized -> cosine via dot product
        show_progress_bar=False,
    )
    return embs

# 6.1 Precompute product embeddings (train universe)
prod_group = esci_train.groupby("product_id")["product_text_clean"].first()
product_ids = prod_group.index.to_list()
prod_embs   = encode_texts(prod_group.tolist(), batch_size=64)

prodid_to_emb = {pid: emb for pid, emb in zip(product_ids, prod_embs)}

# ---- SAVE product embeddings + IDs ----
np.save(PATH_PROD_EMBS, prod_embs)
np.save(PATH_PROD_IDS, np.array(product_ids))

print("Saved product embeddings to:", PATH_PROD_EMBS)
print("Saved product IDs to:", PATH_PROD_IDS)


# 6.2 Compute query embeddings for train/valid (correctly aligned)
train_q = train_df.groupby("query_id")["query"].first().reset_index()
valid_q = valid_df.groupby("query_id")["query"].first().reset_index()

train_query_embs = encode_texts(train_q["query"].tolist())
valid_query_embs = encode_texts(valid_q["query"].tolist())

qid_to_emb_train = {
    int(qid): emb for qid, emb in zip(train_q["query_id"].tolist(), train_query_embs)
}
qid_to_emb_valid = {
    int(qid): emb for qid, emb in zip(valid_q["query_id"].tolist(), valid_query_embs)
}


def attach_gte_scores(df: pd.DataFrame, qid_to_emb: dict) -> pd.DataFrame:
    df = df.copy()
    scores = []
    for idx, row in df.iterrows():
        q_emb = qid_to_emb[int(row["query_id"])]
        p_emb = prodid_to_emb[row["product_id"]]
        score = float(np.dot(q_emb, p_emb))  # cosine because normalized
        scores.append(score)

    df["gte_score"] = scores
    df["gte_rank"] = (
        df.groupby("query_id")["gte_score"]
        .rank(ascending=False, method="first")
        .astype("int32")
    )
    return df

train_df = attach_gte_scores(train_df, qid_to_emb_train)
valid_df = attach_gte_scores(valid_df, qid_to_emb_valid)

train_df[["query_id", "gte_score", "gte_rank"]].head()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration.py: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Saved product embeddings to: esci_pipeline/artifacts_ru/prod_embs.npy
Saved product IDs to: esci_pipeline/artifacts_ru/product_ids.npy


,query_id,gte_score,gte_rank
0,8677,0.486179,1
1,12307,0.631867,2
2,66517,0.615688,1
3,83198,0.548880,2
4,79687,0.422640,4


# 7. RRF Fusion Baseline (BM25 + GTE)

In [22]:
def rrf_fusion(bm25_rank, gte_rank, c: float = 60.0):
    return 1.0 / (c + bm25_rank) + 1.0 / (c + gte_rank)

for df in (train_df, valid_df):
    df["rrf_score"] = rrf_fusion(df["bm25_rank"], df["gte_rank"])
    df["rrf_rank"]  = (
        df.groupby("query_id")["rrf_score"]
        .rank(ascending=False, method="first")
        .astype("int32")
    )

valid_df[["query_id", "rrf_score", "rrf_rank"]].head()



,query_id,rrf_score,rrf_rank
0,14164,0.031778,2
1,45325,0.032266,1
2,52411,0.032258,2
3,63949,0.032002,3
4,100206,0.032522,1


# 8. MMR Baseline (Using GTE embeddings)

In [23]:
def mmr_rerank(df_q: pd.DataFrame, lambda_: float = 0.7, top_k: int = 20) -> pd.DataFrame:
    df_q = df_q.copy()
    cand_idx = df_q.index.to_list()
    selected = []
    remaining = cand_idx.copy()

    # Pre-collect product embeddings
    pid_to_emb_local = {
        pid: prodid_to_emb[pid] for pid in df_q["product_id"].unique()
    }

    while remaining and len(selected) < top_k:
        best_i = None
        best_score = -1e9
        for i in remaining:
            row = df_q.loc[i]
            rel = float(row["gte_score"])
            if not selected:
                score = rel
            else:
                emb_i = pid_to_emb_local[row["product_id"]]
                max_sim = max(
                    float(np.dot(emb_i, pid_to_emb_local[df_q.loc[j, "product_id"]]))
                    for j in selected
                )
                score = lambda_ * rel - (1.0 - lambda_) * max_sim
            if score > best_score:
                best_score = score
                best_i = i
        selected.append(best_i)
        remaining.remove(best_i)

    # Build rank series for selected items
    mmr_order = pd.Series(range(1, len(selected) + 1), index=selected)
    df_q["mmr_rank"] = (
        df_q.index.to_series().map(mmr_order).fillna(len(df_q) + 1).astype("int32")
    )
    return df_q

def apply_mmr(df: pd.DataFrame, lambda_: float = 0.7, top_k: int = 20) -> pd.DataFrame:
    parts = []
    for qid, sub in df.groupby("query_id"):
        parts.append(mmr_rerank(sub, lambda_=lambda_, top_k=top_k))
    return pd.concat(parts, axis=0).sort_index()

train_df = apply_mmr(train_df, lambda_=0.7, top_k=20)
valid_df = apply_mmr(valid_df, lambda_=0.7, top_k=20)

# Convert MMR ranks into scores for evaluation
for df in (train_df, valid_df):
    # Lower rank = better item => use negative rank as score
    df["mmr_score"] = -df["mmr_rank"].astype("float32")


valid_df[["query_id", "mmr_rank"]].head()


,query_id,mmr_rank
0,14164,4
1,45325,3
2,52411,2
3,63949,4
4,100206,2


# 9. LambdaMART (Context-Aware LTR)

In [24]:
# IMPORTANT: sort by query_id so LightGBM group sizes match row order
train_df = train_df.sort_values("query_id").reset_index(drop=True)
valid_df = valid_df.sort_values("query_id").reset_index(drop=True)

ltr_features = [
    "bm25_score",
    "gte_score",
    "ctx_title_len",
    "ctx_desc_len",
    "ctx_bullet_len",
    "ctx_brand_in_title",
    "ctx_color_in_title",
    "prod_count",
    "prod_mean_rel",
    "prod_max_rel",
    "query_count",
    "query_mean_rel",
]

X_train = train_df[ltr_features].astype("float32")
y_train = train_df["relevance"].astype("float32")
group_train = train_df.groupby("query_id").size().to_list()

X_valid = valid_df[ltr_features].astype("float32")
y_valid = valid_df["relevance"].astype("float32")
group_valid = valid_df.groupby("query_id").size().to_list()

lgb_train = lgb.Dataset(X_train, label=y_train, group=group_train)
lgb_valid = lgb.Dataset(X_valid, label=y_valid, group=group_valid, reference=lgb_train)

params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [10],
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_data_in_leaf": 50,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbosity": -1,
    "random_state": RANDOM_STATE,
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=False),
]

lgbm_model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_train, lgb_valid],
    valid_names=["train", "valid"],
    num_boost_round=300,
    callbacks=callbacks,
)

# ---- SAVE LTR model ----
lgbm_model.save_model(str(PATH_LGB_MODEL))
print("Saved LightGBM LambdaMART model to:", PATH_LGB_MODEL)

# 9.1 Predict LTR scores
valid_df["ltr_score"] = lgbm_model.predict(
    X_valid,
    num_iteration=lgbm_model.best_iteration
)
valid_df["ltr_rank"]  = (
    valid_df.groupby("query_id")["ltr_score"]
    .rank(ascending=False, method="first")
    .astype("int32")
)

# ---- SAVE prepared train/valid DataFrames ----
train_df.to_parquet(PATH_TRAIN_PREP, index=False)
valid_df.to_parquet(PATH_VALID_PREP, index=False)

print("Saved prepared train_df to:", PATH_TRAIN_PREP)
print("Saved prepared valid_df to:", PATH_VALID_PREP)


valid_df[["query_id", "ltr_score", "ltr_rank"]].head()


Saved LightGBM LambdaMART model to: esci_pipeline/artifacts_ru/ltr_model_us.txt
Saved prepared train_df to: esci_pipeline/artifacts_ru/train_df_prepared.parquet
Saved prepared valid_df to: esci_pipeline/artifacts_ru/valid_df_prepared.parquet


,query_id,ltr_score,ltr_rank
0,8,0.927398,1
1,70,0.651152,1
2,70,-0.918506,3
3,70,-0.918506,4
4,70,-0.918506,5


# 10. Cross-Encoder (DistilBERT)

In [25]:
def make_hf_dataset(df: pd.DataFrame) -> Dataset:
    tmp = df[["query", "product_text_clean", "relevance"]].rename(
        columns={"relevance": "label"}
    )
    # preserve_index=False avoids keeping the Pandas index as a separate column
    return Dataset.from_pandas(tmp, preserve_index=False)

train_hf = make_hf_dataset(train_df)
valid_hf = make_hf_dataset(valid_df)

tokenizer_x = AutoTokenizer.from_pretrained(XENC_MODEL_NAME)

def tokenize_batch(batch):
    enc = tokenizer_x(
        batch["query"],
        batch["product_text_clean"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )
    enc["labels"] = batch["label"]
    return enc

train_tok = train_hf.map(tokenize_batch, batched=True)
valid_tok = valid_hf.map(tokenize_batch, batched=True)

train_tok = train_tok.remove_columns(["query", "product_text_clean"])
valid_tok = valid_tok.remove_columns(["query", "product_text_clean"])

train_tok.set_format("torch")
valid_tok.set_format("torch")

# 10.1 Model (4-class classification)
num_labels = 4
xenc_model = AutoModelForSequenceClassification.from_pretrained(
    XENC_MODEL_NAME,
    num_labels=num_labels,
    problem_type="single_label_classification",
)

training_args = TrainingArguments(
    output_dir="models/esci_ru_xenc",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = (preds == labels).mean()
    return {"accuracy": float(acc)}

trainer = Trainer(
    model=xenc_model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=valid_tok,
    compute_metrics=compute_metrics,
)


trainer.train()

# 10.2 Getting CE scores for valid_df after training:
preds = trainer.predict(valid_tok)
logits = preds.predictions
probs  = torch.softmax(torch.tensor(logits), dim=-1).numpy()
valid_df["ce_score"] = probs[:, 3] + 0.5 * probs[:, 2]  # weight Exact/Substitute
valid_df["ce_rank"]  = (
    valid_df.groupby("query_id")["ce_score"]
    .rank(ascending=False, method="first")
    .astype("int32")
)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/107125 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/542M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: karych1807 (karych1807-university-of-vienna) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,1.142600,1.203222,0.459879
2,0.983900,1.241358,0.465251


# 11. Evaluation (nDCG@K, P@K, gAUC)

In [26]:
def dcg_at_k(rels, k: int):
    rels = np.asarray(rels)[:k]
    if rels.size == 0:
        return 0.0
    discounts = np.log2(np.arange(2, rels.size + 2))
    return float(np.sum((2 ** rels - 1) / discounts))

def ndcg_at_k(rels, k: int):
    rels = np.asarray(rels)
    best = np.sort(rels)[::-1]
    best_dcg = dcg_at_k(best, k)
    if best_dcg == 0:
        return 0.0
    return dcg_at_k(rels, k) / best_dcg

def eval_ranking(df: pd.DataFrame, score_col: str, k: int = 10):
    ndcgs = []
    precs = []
    auc_labels = []
    auc_scores = []

    for qid, sub in df.groupby("query_id"):
        sub_sorted = sub.sort_values(score_col, ascending=False)
        rels = sub_sorted["relevance"].values

        ndcgs.append(ndcg_at_k(rels, k))
        precs.append((rels[:k] > 0).mean())  # non-zero relevance => positive

        auc_labels.extend((rels > 0).astype(int).tolist())
        auc_scores.extend(sub_sorted[score_col].tolist())

    ndcg = float(np.mean(ndcgs))
    prec = float(np.mean(precs))
    try:
        gauc = float(roc_auc_score(auc_labels, auc_scores))
    except ValueError:
        gauc = float("nan")

    return {"nDCG@{}".format(k): ndcg, "P@{}".format(k): prec, "gAUC": gauc}

for col in ["bm25_score", "gte_score", "rrf_score", "ltr_score","mmr_score", "ce_score"]:
    metrics = eval_ranking(valid_df, col, k=10)
    print(col, metrics)


bm25_score {'nDCG@10': 0.8219233391832647, 'P@10': 0.7967030234930967, 'gAUC': 0.5613473469004544}
gte_score {'nDCG@10': 0.8341365994421416, 'P@10': 0.7970284913531456, 'gAUC': 0.6730971647964256}
rrf_score {'nDCG@10': 0.8089144842989672, 'P@10': 0.7963978973743009, 'gAUC': 0.5501003768398196}
ltr_score {'nDCG@10': 0.8757313348555434, 'P@10': 0.7974963514019657, 'gAUC': 0.9647975168061643}
mmr_score {'nDCG@10': 0.8339985160560252, 'P@10': 0.7967437069756029, 'gAUC': 0.5851658804968771}
ce_score {'nDCG@10': 0.8304700573740978, 'P@10': 0.7966826817518436, 'gAUC': 0.6081213404559338}
